# <u> Copulax Examples </u>

## Multivariate Distributions

CopulAX provides a number of multivariate distribution objects, a full list of which can be found <a href=https://github.com/tfm000/copulax/blob/main/copulax/multivariate/README.md> here</a>.
These distribution objects contain standardised methods, covering almost all intended usecases. Inspection of each object also allows the user to see the implemented parameterisation and other details.

### Parameter Specification

All copulAX distribution objects utilise python dictionaries to label and hold parameters.

Each distribution object implements the `example_params` method, allowing the user to quickly and easily get a sense of what the required parameter key-value naming and form.

In [13]:
from copulax.multivariate import mvt_student_t

print(mvt_student_t.example_params())

{'nu': Array(2.5, dtype=float32), 'mu': Array([[0.],
       [0.],
       [0.]], dtype=float32), 'sigma': Array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)}


### Density Functions

All multivariate distribution objects have `pdf` and `logpdf` methods for evaluating the (log) probability density function.

In [14]:
# generating a random sample
import numpy as np
dim = 3
sample = np.random.normal(loc=0, scale=1, size=(100, dim)) * np.random.standard_t(df=5, size=(100, dim))

# calculating the PDF and log-PDF
example_params = mvt_student_t.example_params(dim=dim)
pdf = mvt_student_t.pdf(sample, params=example_params)
logpdf = mvt_student_t.logpdf(sample, params=example_params)
print("pdf:", pdf[:5])
print("logpdf:", logpdf[:5])

pdf: [[0.07171081]
 [0.04073118]
 [0.00028424]
 [0.06138092]
 [0.07281803]]
logpdf: [[-2.6351137]
 [-3.2007613]
 [-8.1656885]
 [-2.7906563]
 [-2.6197917]]


### Generating Random Samples

All copulAX distribution objects are capable of generating random samples using the `rvs` method.
As copulAX is JAX based, a key is required for random number generation. 
Random keys can be generated using copulAX's `get_random_key` function, as shown below.


In [15]:
# generating a random key
from copulax import get_random_key
from jax.random import split
key = get_random_key()
key, subkey = split(key)

# generating a random sample
random_sample = mvt_student_t.rvs(key=subkey, params=example_params, size=100)
print("random sample:", random_sample[:5])

random sample: [[-2.006089    2.5706308  -0.14923695]
 [-0.09261097  0.1776195   0.5453869 ]
 [ 0.7623694   1.4573921   4.714453  ]
 [ 0.6908576   0.4484164   1.7388321 ]
 [ 3.9341643  -4.5449767   1.0256963 ]]


### Fitting Distributions to Data

All copulAX distributions are capable of fitting parameters to a given set of observations using the `fit` method.

In [21]:
fitted_mvt_t = mvt_student_t.fit(sample)
print("fitted parameters:", fitted_mvt_t.params)

fitted parameters: {'nu': Array(2.5446897, dtype=float32), 'mu': Array([[ 0.0444343 ],
       [-0.14779878],
       [ 0.07223894]], dtype=float32), 'sigma': Array([[ 0.4776612 , -0.07564374, -0.04871784],
       [-0.07564374,  0.64304864,  0.01440321],
       [-0.04871784,  0.01440321,  0.24687159]], dtype=float32)}


### Distribution Statistics

The `stats` method computes key statistics (mean, median, mode, covariance, skewness) from the distribution parameters.

In [24]:
stats = fitted_mvt_t.stats()
for stat_name, stat_val in stats.items():
    print(f"{stat_name}:\n {stat_val}")

mean:
 [[ 0.0444343 ]
 [-0.14779878]
 [ 0.07223894]]
median:
 [[ 0.0444343 ]
 [-0.14779878]
 [ 0.07223894]]
mode:
 [[ 0.0444343 ]
 [-0.14779878]
 [ 0.07223894]]
cov:
 [[ 2.2315452  -0.3533936  -0.22760078]
 [-0.3533936   3.0042048   0.06728914]
 [-0.22760078  0.06728914  1.1533387 ]]
skewness:
 [[0.]
 [0.]
 [0.]]


### Correlation and Covariance Utilities

CopulAX provides utility functions for generating and working with correlation/covariance matrices.

In [18]:
from copulax.multivariate import corr, cov, random_correlation, random_covariance
from copulax.univariate import uniform
from copulax import get_random_key
from jax.random import split
import jax.numpy as jnp

key = get_random_key()

# generate a random correlation matrix
key1, _subkey = split(key)
key2, key3 = split(_subkey)
R = random_correlation(size=dim, key=key1)
print("Random correlation matrix:\n", R)

# generate a random covariance matrix
random_vars = uniform.rvs(params={"a": 0.5, "b": 2.0}, size=(dim,)) # required for covariance matrix generation
C = random_covariance(vars=random_vars, key=key)
print("Random covariance matrix:\n", C)

# compute correlation and covariance from data
data_corr = corr(jnp.array(sample))
data_cov = cov(jnp.array(sample))
print("Data correlation:\n", data_corr)
print("Data covariance:\n", data_cov)

Random correlation matrix:
 [[ 0.99999994  0.29167935 -0.09186094]
 [ 0.29167935  1.0000001   0.36216447]
 [-0.09186094  0.36216447  1.        ]]
Random covariance matrix:
 [[ 0.65645957 -0.37627313 -0.35806325]
 [-0.37627313  1.0980053   0.5341063 ]
 [-0.35806325  0.5341064   0.79110867]]
Data correlation:
 [[ 1.         -0.13648693 -0.14187063]
 [-0.13648693  1.          0.03614946]
 [-0.14187063  0.03614946  1.        ]]
Data covariance:
 [[ 2.231545   -0.3533936  -0.22760077]
 [-0.35339358  3.0042048   0.06728914]
 [-0.22760077  0.06728914  1.1533386 ]]


### JIT Compilation

Most evaluation methods are compatible with JAX's `jit` for accelerated computation.

In [19]:
from jax import jit

# JIT-compiled log-PDF
jit_logpdf = jit(mvt_student_t.logpdf)
jit_result = jit_logpdf(sample, params=example_params)
print("JIT logpdf:", jit_result[:5])

# JIT-compiled sampling
jit_rvs = jit(mvt_student_t.rvs, static_argnums=(0,))
key, subkey = split(key)
jit_sample = jit_rvs(100, example_params, subkey)
print("JIT sample shape:", jit_sample.shape)

JIT logpdf: [[-2.635114 ]
 [-3.2007616]
 [-8.1656885]
 [-2.7906566]
 [-2.619792 ]]
JIT sample shape: (100, 3)


### Saving and Loading Fitted Distributions

Fitted distributions can be saved to disk and loaded back in a later session using the `.save()` method and `copulax.load()` function. Files use the `.cpx` format and are cross-platform (Windows, macOS, Linux).

In [25]:
import copulax

# save the fitted multivariate distribution
fitted_mvt_t.save("fitted_mvt_student_t.cpx")

# load it back
loaded = copulax.load("fitted_mvt_student_t.cpx")
print("loaded parameters:", loaded.params)
print("logpdf matches:", jnp.allclose(
    mvt_student_t.logpdf(sample[:5], params=fitted_mvt_t.params),
    loaded.logpdf(sample[:5])
))

# clean up
import os
os.remove("fitted_mvt_student_t.cpx")

loaded parameters: {'nu': Array(2.5446897, dtype=float32), 'mu': Array([[ 0.0444343 ],
       [-0.14779878],
       [ 0.07223894]], dtype=float32), 'sigma': Array([[ 0.4776612 , -0.07564374, -0.04871784],
       [-0.07564374,  0.64304864,  0.01440321],
       [-0.04871784,  0.01440321,  0.24687159]], dtype=float32)}
logpdf matches: True
